# ML Model Comparison — Rolling Backtest

Trains several ML models (and the project's own baselines) to predict each weather variable, using a **walk-forward backtest**: train on an expanding window of past days, predict the next day, then move the window forward one day (now including that day's real outcome) and repeat. This is the "proper backtest comparing ML vs best single provider vs simple average vs weighted average" listed as a future upgrade in the README.

Data: the 3 backfilled Open-Meteo models (`ecmwf_ifs025`, `gfs_global`, `gem_seamless`) joined against actuals — these are the only sources with enough paired history to backtest meaningfully. Roughly the first 2 months seed the earliest test day's training window; the final ~1 month is walked forward day by day as the evaluation period.

Select the **Weather Ensemble (.venv)** kernel before running. The main backtest cell retrains every model at every step, so it takes roughly 2 minutes.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.pipeline import Pipeline

from weather_ensemble.config import OPEN_METEO_BACKFILL_MODELS, get_default_location
from weather_ensemble.ml import _build_wide_feature_table
from weather_ensemble.service import load_modelling_table

DB_PATH = Path("../data/weather.db") if Path("../data/weather.db").exists() else Path("data/weather.db")
LOCATION = get_default_location()
SOURCES = [f"open_meteo_{model}" for model in OPEN_METEO_BACKFILL_MODELS]

print(f"Location: {LOCATION.name}")
print(f"Sources used: {SOURCES}")

Location: Melbourne
Sources used: ['open_meteo_ecmwf_ifs025', 'open_meteo_gfs_global', 'open_meteo_gem_seamless']


## Load the 3 backfilled sources + actuals, build the wide feature table

Only rows from `SOURCES` are kept before pivoting, so the source-agreement stats (mean/std/etc.) reflect exactly these 3 models — not any live-only source that might also be sitting in the database.

In [2]:
long_df = load_modelling_table(DB_PATH, LOCATION)
long_df = long_df[long_df["source"].isin(SOURCES)]

wide_df = _build_wide_feature_table(long_df, include_targets=True)
wide_df = wide_df.sort_values("forecast_date").reset_index(drop=True)

print(f"{len(wide_df)} days available, {wide_df['forecast_date'].min().date()} to {wide_df['forecast_date'].max().date()}")
wide_df.head()

90 days available, 2026-04-04 to 2026-07-02


,location_name,forecast_date,actual_max_temp,actual_min_temp,actual_precipitation_sum,actual_did_rain,actual_uv_index,actual_wind_speed,actual_wind_gusts,open_meteo_ecmwf_ifs025__max_temp,...,source_max__pressure_msl,source_range__pressure_msl,source_count__pressure_msl,source_mean__weather_code,source_median__weather_code,source_std__weather_code,source_min__weather_code,source_max__weather_code,source_range__weather_code,source_count__weather_code
0,Melbourne,2026-04-04,19.0,10.1,0.0,0,None,14.7,38.9,19.2,...,1026.913,0.521,3,2.666667,3.0,0.471405,2.0,3.0,1.0,3
1,Melbourne,2026-04-05,25.3,8.5,0.0,0,None,11.7,22.7,25.6,...,1022.433,0.779,3,2.666667,3.0,0.471405,2.0,3.0,1.0,3
2,Melbourne,2026-04-06,22.2,14.2,14.4,1,None,15.1,37.4,21.2,...,1015.463,0.367,3,55.000000,53.0,4.320494,51.0,61.0,10.0,3
3,Melbourne,2026-04-07,19.5,14.1,2.9,1,None,20.3,44.3,19.5,...,1013.133,0.262,3,51.000000,51.0,0.000000,51.0,51.0,0.0,3
4,Melbourne,2026-04-08,18.8,10.4,1.5,1,None,12.5,28.8,19.3,...,1015.754,0.125,3,19.000000,3.0,22.627417,3.0,51.0,48.0,3


### Actual-data coverage check

A target with zero non-null actuals can't be backtested at all — it'll be silently skipped further down, so surface that here instead of leaving it unexplained.

In [3]:
from weather_ensemble.config import TARGETS

print(f"Non-null actual coverage per target (out of {len(wide_df)} days):")
for target in TARGETS:
    col = f"actual_{target}"
    count = int(wide_df[col].notna().sum()) if col in wide_df.columns else 0
    flag = "  <-- no observed data; will be skipped in the backtest below" if count == 0 else ""
    print(f"  {target:20s} {count:3d}{flag}")

Non-null actual coverage per target (out of 90 days):
  max_temp              90
  min_temp              90
  precipitation_sum     90
  did_rain              90
  uv_index               0  <-- no observed data; will be skipped in the backtest below
  wind_speed            90
  wind_gusts            90


## Train/test split — ~2 months initial train, ~1 month rolling test

In [4]:
TRAIN_FRACTION = 2 / 3  # ~2 months of a ~3-month dataset

train_size = int(round(len(wide_df) * TRAIN_FRACTION))
test_size = len(wide_df) - train_size

print(f"Initial training window: {train_size} days "
      f"({wide_df['forecast_date'].iloc[0].date()} to {wide_df['forecast_date'].iloc[train_size - 1].date()})")
print(f"Rolling test window: {test_size} days "
      f"({wide_df['forecast_date'].iloc[train_size].date()} to {wide_df['forecast_date'].iloc[-1].date()})")

Initial training window: 60 days (2026-04-04 to 2026-06-02)
Rolling test window: 30 days (2026-06-03 to 2026-07-02)


## Candidates

- **Baselines** mirror what's already in production (`service.py`): `simple_average` (unweighted mean across the 3 sources), `best_single_source` (whichever source had the lowest MAE on the training window so far), and `mae_weighted_average` (the same inverse-MAE weighting `blend_forecast` uses today).
- **ML models**: Linear Regression, Ridge, Random Forest, Gradient Boosting, Hist Gradient Boosting — each retrained from scratch at every step on all data available up to that point.

In [5]:
CONTINUOUS_TARGETS = ["max_temp", "min_temp", "precipitation_sum", "uv_index", "wind_speed", "wind_gusts"]


def compute_source_maes(train_df: pd.DataFrame, var: str) -> dict:
    """Per-source MAE for `var`, computed only from the training window (no lookahead)."""
    actual_col = f"actual_{var}"
    maes = {}
    for source in SOURCES:
        col = f"{source}__{var}"
        if col not in train_df or actual_col not in train_df:
            continue
        pair = train_df[[col, actual_col]].dropna()
        if not pair.empty:
            maes[source] = (pair[col] - pair[actual_col]).abs().mean()
    return maes


def baseline_predictions(test_row: pd.DataFrame, var: str, maes: dict) -> dict:
    """simple_average / best_single_source / mae_weighted_average for one test day."""
    preds = {}

    mean_col = f"source_mean__{var}"
    if mean_col in test_row and pd.notna(test_row[mean_col].iloc[0]):
        preds["simple_average"] = float(test_row[mean_col].iloc[0])

    available = {s: test_row[f"{s}__{var}"].iloc[0] for s in SOURCES if f"{s}__{var}" in test_row}
    available = {s: v for s, v in available.items() if pd.notna(v)}
    if not available:
        return preds

    if maes:
        ranked = sorted(maes, key=maes.get)
        best_source = next((s for s in ranked if s in available), None)
        if best_source:
            preds["best_single_source"] = float(available[best_source])

        weighted_sum, total_weight = 0.0, 0.0
        for source, value in available.items():
            mae = maes.get(source)
            weight = 1.0 / mae if mae and mae > 0 else 1.0
            weighted_sum += value * weight
            total_weight += weight
        preds["mae_weighted_average"] = weighted_sum / total_weight
    else:
        preds["best_single_source"] = float(next(iter(available.values())))
        preds["mae_weighted_average"] = float(np.mean(list(available.values())))

    return preds

### Feature set — narrowed per target to curb overfitting

Using all ~111 engineered columns (every forecast variable's per-source values and cross-source stats) as
features for every target — against only ~60 training rows at the start of the test window — is a classic
`p >> n` setup, and it showed: Linear Regression in particular overfit badly.

Fixed by restricting each target's features to just its own variable: the 3 raw per-source forecasts, the
cross-source mean/median/std/min/max/range for that variable, and 3 date features (month, day-of-year,
day-of-week). That's ~12 features per target instead of 111 — comfortably below the training window size.
`source_count` is dropped entirely since all 3 sources are always present in this backfilled dataset
(constant value, zero information). The tree-based models also get explicit depth/subsample limits as a
second line of defense against overfitting on a still-small dataset.

In [6]:
DATE_FEATURES = ["month", "day_of_year", "day_of_week"]


def features_for_target(var: str) -> list:
    """Only this variable's own signals - its 3 per-source forecasts, cross-source stats, and date features."""
    cols = [f"{source}__{var}" for source in SOURCES]
    cols += [f"source_{stat}__{var}" for stat in ["mean", "median", "std", "min", "max", "range"]]
    cols = [c for c in cols if c in wide_df.columns]
    return cols + DATE_FEATURES


def make_regressors() -> dict:
    """Fresh, unfitted pipelines — called once per (test day, variable) so no state leaks across fits."""
    return {
        "linear_regression": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", LinearRegression())]),
        "ridge": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", Ridge(alpha=1.0))]),
        "random_forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=200, max_depth=6, min_samples_leaf=3, random_state=42, n_jobs=-1,
            )),
        ]),
        "gradient_boosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingRegressor(n_estimators=100, max_depth=3, subsample=0.8, random_state=42)),
        ]),
        "hist_gradient_boosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(max_depth=4, l2_regularization=1.0, random_state=42)),
        ]),
    }


print("Feature count per target:")
for var in CONTINUOUS_TARGETS:
    print(f"  {var:20s} {len(features_for_target(var))} features")

Feature count per target:
  max_temp             12 features
  min_temp             12 features
  precipitation_sum    12 features
  uv_index             10 features
  wind_speed           12 features
  wind_gusts           12 features


## Run the walk-forward backtest

Retrains every model at every step — takes roughly 2 minutes.

In [7]:
records = []

for i in range(train_size, len(wide_df)):
    train_df = wide_df.iloc[:i]
    test_row = wide_df.iloc[[i]]
    test_date = test_row["forecast_date"].iloc[0]

    for var in CONTINUOUS_TARGETS:
        actual_col = f"actual_{var}"
        if actual_col not in wide_df.columns or pd.isna(test_row[actual_col].iloc[0]):
            continue
        actual_value = float(test_row[actual_col].iloc[0])

        maes = compute_source_maes(train_df, var)
        candidates = baseline_predictions(test_row, var, maes)

        features = features_for_target(var)
        train = train_df[features + [actual_col]].dropna(subset=[actual_col])
        if len(train) >= 10:
            X_train, y_train = train[features], train[actual_col]
            X_test = test_row.reindex(columns=features)
            for name, model in make_regressors().items():
                model.fit(X_train, y_train)
                candidates[name] = float(model.predict(X_test)[0])

        for candidate, prediction in candidates.items():
            if prediction is None or (isinstance(prediction, float) and np.isnan(prediction)):
                continue
            records.append({
                "forecast_date": test_date,
                "variable": var,
                "candidate": candidate,
                "prediction": prediction,
                "actual": actual_value,
                "abs_error": abs(prediction - actual_value),
            })

    done = i - train_size + 1
    if done % 5 == 0 or done == test_size:
        print(f"  ...evaluated {done}/{test_size} test days")

results_df = pd.DataFrame(records)
print(f"\n{len(results_df)} (day, variable, candidate) predictions scored")

  ...evaluated 5/30 test days


  ...evaluated 10/30 test days


  ...evaluated 15/30 test days


  ...evaluated 20/30 test days


  ...evaluated 25/30 test days


  ...evaluated 30/30 test days

1200 (day, variable, candidate) predictions scored


## Results — MAE per candidate per variable

Lower is better.

In [8]:
summary = results_df.groupby(["variable", "candidate"])["abs_error"].mean().unstack("candidate").round(3)
summary

candidate,best_single_source,gradient_boosting,hist_gradient_boosting,linear_regression,mae_weighted_average,random_forest,ridge,simple_average
variable,,,,,,,,
max_temp,0.363,0.535,0.852,0.348,0.472,0.447,0.338,0.550
min_temp,0.267,0.376,0.743,0.244,0.357,0.438,0.249,0.509
precipitation_sum,0.827,1.501,2.098,1.141,1.087,1.688,1.080,1.233
wind_gusts,3.407,2.905,4.606,2.061,3.240,3.263,2.085,3.819
wind_speed,1.857,1.404,1.723,1.141,1.265,1.150,1.059,1.399


## Overall ranking

Raw MAE isn't comparable across variables on different scales (°C vs km/h vs mm), so rank candidates within each variable, then average the ranks — a rank of 1 means "best on average across all variables."

In [9]:
ranks = summary.rank(axis=1)
overall_rank = ranks.mean(axis=0).sort_values()
print("Average rank across all variables (lower is better):")
overall_rank

Average rank across all variables (lower is better):


candidate
ridge                     1.6
linear_regression         2.0
mae_weighted_average      4.0
best_single_source        4.2
random_forest             5.0
gradient_boosting         5.2
simple_average            6.2
hist_gradient_boosting    7.8
dtype: float64

In [10]:
best = overall_rank.index[0]
print(f"Best-performing candidate over this backtest: '{best}'\n")

if "mae_weighted_average" in summary.columns and best != "mae_weighted_average":
    print("How it compares to the project's current production baseline ('mae_weighted_average'):")
    display(summary[[best, "mae_weighted_average"]])
else:
    display(summary[[best]])

Best-performing candidate over this backtest: 'ridge'

How it compares to the project's current production baseline ('mae_weighted_average'):


candidate,ridge,mae_weighted_average
variable,,
max_temp,0.338,0.472
min_temp,0.249,0.357
precipitation_sum,1.080,1.087
wind_gusts,2.085,3.240
wind_speed,1.059,1.265


## Bonus: does it rain? (classification)

`did_rain` is binary, so it needs classifiers rather than regressors. Compared here: a majority-class baseline, logistic regression, and a random forest classifier — all restricted to `features_for_target("precipitation_sum")` since rain is fundamentally a precipitation question, not a humidity/pressure/wind one.

In [11]:
def make_classifiers() -> dict:
    return {
        "logistic_regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")), ("model", LogisticRegression(max_iter=1000)),
        ]),
        "random_forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=3, random_state=42, n_jobs=-1)),
        ]),
    }


clf_features = features_for_target("precipitation_sum")
clf_records = []
actual_col = "actual_did_rain"

for i in range(train_size, len(wide_df)):
    train_df = wide_df.iloc[:i]
    test_row = wide_df.iloc[[i]]
    test_date = test_row["forecast_date"].iloc[0]

    if actual_col not in wide_df.columns or pd.isna(test_row[actual_col].iloc[0]):
        continue
    actual_value = int(test_row[actual_col].iloc[0])

    train = train_df[clf_features + [actual_col]].dropna(subset=[actual_col])
    if len(train) < 10 or train[actual_col].nunique() < 2:
        continue

    X_train, y_train = train[clf_features], train[actual_col]
    X_test = test_row.reindex(columns=clf_features)

    predictions = {"majority_class": int(y_train.mode().iloc[0])}
    for name, model in make_classifiers().items():
        model.fit(X_train, y_train)
        predictions[name] = int(model.predict(X_test)[0])

    for candidate, prediction in predictions.items():
        clf_records.append({
            "forecast_date": test_date,
            "candidate": candidate,
            "prediction": prediction,
            "actual": actual_value,
            "correct": prediction == actual_value,
        })

clf_results_df = pd.DataFrame(clf_records)
print(f"{clf_results_df['forecast_date'].nunique()} test days scored for did_rain\n")
clf_results_df.groupby("candidate")["correct"].mean().sort_values(ascending=False).rename("accuracy")

30 test days scored for did_rain



candidate
logistic_regression    0.866667
random_forest          0.866667
majority_class         0.400000
Name: accuracy, dtype: float64

## Your own experiments

Adjust `TRAIN_FRACTION`, `CONTINUOUS_TARGETS`, `features_for_target()`, or the model dicts in `make_regressors()` / `make_classifiers()` above and re-run the backtest cells to explore further.